# pandas: `DataFrame` column merging/joining

In an earlier lesson I discussed how to concatenate `DataFrame` objects using the `pd.concat()` function. You can also add columns to a `DataFrame` utilizing the [`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) function or the [`DataFrame.merge()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html) and [`DataFrame.join()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html) methods. In this lesson I'll demonstrate how to leverage these [callables](https://docs.python.org/3/glossary.html#term-callable) to perform SQL-like join operations.

In [1]:
import numpy as np
import pandas as pd

## `DataFrame` column merge/join operations

Utilize [`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) or [`DataFrame.merge()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html) for **column-on-column** operations. Both callables mimic database-style join operations, including one-to-one, many-to-one, and many-to-many joins. If you need to join **index-on-index** consider the [`DataFrame.join()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html) method.

`pandas.merge(left, right, how='inner', on=None, left_on=None, right_on=None, left_index=False, right_index=False, sort=False, suffixes=('_x', '_y'), copy=None, indicator=False, validate=None)`

`DataFrame.merge(right, how='inner', on=None, left_on=None, right_on=None, left_index=False, right_index=False, sort=False, suffixes=('_x', '_y'), copy=None, indicator=False, validate=None)`

`DataFrame.join(other, on=None, how='left', lsuffix='', rsuffix='', sort=False, validate=None)`

All three callables include a `how=< str >` parameter that specifies the type of join to perform&mdash;"left", "right", "outer", "inner", or "cross"&mdash;while other parameters define the index or column(s) to join on, suffixes to append to overlapping column labels, etc.

## Join types

For those unfamiliar with relational database querying and the structured query language (SQL), the various join types that pandas mimics are summarized below.

![Join types: left, right, inner, outer, and cross](img/join_types-w600.png)

| Type | Key Usage | Result |
|:---- | :-------- | :------ |
| left | Left frame keys only | Selects all rows from the left frame; unmatched rows from the right frame are discarded. |
| right | Right frame keys only. | Selects all rows from the right frame; unmatched rows from the left frame are discarded. |
| inner | Intersection of keys from both frames. | Selects rows _shared_ between the two frames; unmatched rows from either frame are discarded. |
| outer | Union of keys from both frames. | Selects all rows from both frames; `Nan` values are inserted into unmatched rows. |
| cross | Cartesian product of both frames. | Each row from the left frame is paired with each row from the right frame. |

Both `pd.merge()` and `DataFrame.merge()` perform an **inner join** by default while `DataFrame.join()` defaults to a **left join**. I focus on these join types in the examples below, leaving the right, outer, and cross joins for you to explore in the exercises.

## Merge/Join operations

You can add the columns of one `DataFrame` to another by specifying the join type and joining the frames either _index-on-index_ or _column(s)-on-column(s)_.

To illustrate these operations I'll create two "toy" dataframes comprising two columns: "group" and "letters". `letters_1` features three rows while `letters_2` is limited to two rows.

In [2]:
letters_1 = pd.DataFrame({"group": [1, 2, 3], "letters": ["P Q Y P Q", "E W L N O", "J U J G V"]})
letters_1.shape

(3, 2)

In [3]:
letters_2 = pd.DataFrame({"group": [2, 3], "letters": ["N H X O G", "W F J T S"]})
letters_2.shape

(2, 2)

## index-on-index

### Inner join

As noted earlier, both `pd.merge()` and `DataFrame.merge()` default to an **inner join** (`how=inner`). However, to join index-on-index I will need to pass to each callable the keyword arguments `left_index=True` and `right_index=True`. Both callables also provide default column label suffixes (`suffixes=("_x", "_y")`) that will be appended to otherwise overlapping column labels.

In contrast, `DataFrame.join()` defaults to a **left join** so the keyword argument `how=inner` _must_ be passed explicitly. The `join()` method is optimized for index-on-index joining so instructing the method to join on the indices is not required.

The result of this operation is a new `DataFrame`.

💡 I can also pass the keyword argument `indicator=True` to `pd.merge()` and `DataFrame.merge()` to add an additional column named "_merge" that indicates the source of each row. Adding this column is optional and for observational purposes only.

#### `pd.merge()` / `DataFrame.merge()`

In [4]:
result = pd.merge(letters_1, letters_2, left_index=True, right_index=True, indicator=True)
result

,group_x,letters_x,group_y,letters_y,_merge
0,1,P Q Y P Q,2,N H X O G,both
1,2,E W L N O,3,W F J T S,both


In [5]:
result = letters_1.merge(letters_2, left_index=True, right_index=True, indicator=True)
result

,group_x,letters_x,group_y,letters_y,_merge
0,1,P Q Y P Q,2,N H X O G,both
1,2,E W L N O,3,W F J T S,both


#### `DataFrame.join()`

In [6]:
result = letters_1.join(letters_2, how="inner", lsuffix="_x", rsuffix="_y")
result

,group_x,letters_x,group_y,letters_y
0,1,P Q Y P Q,2,N H X O G
1,2,E W L N O,3,W F J T S


### Left join

Next, let's join the columns in `letters_2` to `letters_1` via a __left join__ on the indices. All `letters_1` rows are returned along with matching `letters_2` rows. A `NaN` value is inserted into the unmatched row.

Just for fun, I'll override the default suffix values with "_1" and "_2".


#### `pd.merge()` / `DataFrame.merge()`


In [7]:
result = pd.merge(
    letters_1,
    letters_2,
    how="left",
    left_index=True,
    right_index=True,
    suffixes=("_1", "_2"),
    indicator=True,
)
result

,group_1,letters_1,group_2,letters_2,_merge
0,1,P Q Y P Q,2.0,N H X O G,both
1,2,E W L N O,3.0,W F J T S,both
2,3,J U J G V,NaN,NaN,left_only


In [8]:
result = letters_1.merge(
    letters_2, how="left", left_index=True, right_index=True, suffixes=("_1", "_2"), indicator=True
)
result

,group_1,letters_1,group_2,letters_2,_merge
0,1,P Q Y P Q,2.0,N H X O G,both
1,2,E W L N O,3.0,W F J T S,both
2,3,J U J G V,NaN,NaN,left_only


#### `DataFrame.join()`

In [9]:
result = letters_1.join(letters_2, lsuffix="_1", rsuffix="_2")
result

,group_1,letters_1,group_2,letters_2
0,1,P Q Y P Q,2.0,N H X O G
1,2,E W L N O,3.0,W F J T S
2,3,J U J G V,NaN,NaN


## column-on-column joins

### Left join

Let's join on the "group" column. For both `pd.merge()` and `DataFrame.merge()` I _must_ set the index of both frames to "group" via the keyword argument `on` and then perform a **left join** on the "group" column.

#### `pd.merge()` / `DataFrame.merge()`

In [10]:
result = pd.merge(
    letters_1, letters_2, how="left", on="group", suffixes=("_1", "_2"), indicator=True
)
result

,group,letters_1,letters_2,_merge
0,1,P Q Y P Q,NaN,left_only
1,2,E W L N O,N H X O G,both
2,3,J U J G V,W F J T S,both


In [11]:
result = letters_1.merge(
    letters_2, how="left", on="group", suffixes=("_1", "_2"), indicator=True
)  # overriding default suffixes
result

,group,letters_1,letters_2,_merge
0,1,P Q Y P Q,NaN,left_only
1,2,E W L N O,N H X O G,both
2,3,J U J G V,W F J T S,both


#### `DataFrame.join()`

Performing a left join on the "group" column is not as straightforward when calling the `DataFrame.join()` method. One option is to set the index of both frames to "group" by passing the column label to the [`DataFrame.set_index()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) method.

In [12]:
result = letters_1.set_index("group").join(letters_2.set_index("group"), lsuffix="_1", rsuffix="_2")
result

,letters_1,letters_2
group,,
1,P Q Y P Q,NaN
2,E W L N O,N H X O G
3,J U J G V,W F J T S


If I prefer to retain the original `letter_1` index while joining on the "group" column, I only need to set the `letter_2` index to "group" as is illustrated below.

In [13]:
result = letters_1.join(letters_2.set_index("group"), on="group", lsuffix="_1", rsuffix="_2")
result

,group,letters_1,letters_2
0,1,P Q Y P Q,NaN
1,2,E W L N O,N H X O G
2,3,J U J G V,W F J T S


## Back to the material world

### Retrieve the data

Let's call the pandas `read_csv()` function and retrieve the 2002-24 Mega Millions lottery data that we combined earlier from four separate CSV files. I'll assign the `DataFrame` returned by the function call to the variable `data`.

In [14]:
data = pd.read_csv("./data/mega_millions_combined-2002_24.csv")

In [15]:
data.head(3)

,draw_date,winning_numbers,mega_ball,multiplier
0,05/17/2024,08 17 40 60 70,3,2.0
1,05/14/2024,13 19 43 62 64,6,3.0
2,05/10/2024,13 22 26 32 65,18,4.0


### Split winning numbers string into numeric columns

Working with a collection of numbers masquerading as a string is always tricky. Such is the case with the "winning numbers" column of strings. An alternative strategy is to split each string into a `Series` that is expanded into a `DataFrame` comprising five numeric (`int64`) columns labeled `Pick5_1` to `Pick5_5`. The resulting `DataFrame` is named `pick5`.

I can then add the `pick5` columns to `data` utilizing either `pd.merge()`, `DataFrame.merge()`, `DataFrame.join()` or `pd.concat()`.

In [16]:
pick5_cols_mapper = {0: "pick5_1", 1: "pick5_2", 2: "pick5_3", 3: "pick5_4", 4: "pick5_5"}
pick5 = (
    data.loc[:, "winning_numbers"]
    .str.split(expand=True)
    .apply(pd.to_numeric)
    .rename(columns=pick5_cols_mapper)
)
pick5.head(3)

,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,8,17,40,60,70
1,13,19,43,62,64
2,13,22,26,32,65


### Combine data row-wise

Let's step through each of the callables and add the `pick5` columns to `data` by joining the frames on their indices. Given that the `pick5` frame was spawned from every row in `data` I'll instruct each callable to perform an **inner join**.

💡 I can also confirm that each of the merge keys is unique by passing the keyword argument `validate="one_to_one"`. The integrity check is optional.

#### `pd.merge()` / `DataFrame.merge()`

In [17]:
data_v1p0 = pd.merge(data, pick5, left_index=True, right_index=True, validate="one_to_one")
data_v1p0.shape

(2294, 9)

In [18]:
data_v1p1 = data.merge(pick5, left_index=True, right_index=True)
data_v1p1.shape

(2294, 9)

#### `DataFrame.join()`

In [19]:
data_v1p2 = data.join(pick5, how="inner", validate="one_to_one")
data_v1p2.shape

(2294, 9)

#### `pd.concat()`

Alternatively, you can call `pd.concat()`, pass it the sequence `[data, pick5]`, and concatatenate two `DataFrame` objects along the column axis (`axis=1` or `axis="columns"`).

In [20]:
data_v1p3 = pd.concat([data, pick5], axis=1, verify_integrity=True)  # optional verify
data_v1p3.shape

(2294, 9)

I can confirm that each of the `data_v1p*` frames are equal by calling [`pd.testing.assert_frame_equal()`](https://pandas.pydata.org/docs/reference/api/pandas.testing.assert_frame_equal.html).

In [21]:
pd.testing.assert_frame_equal(data_v1p0, data_v1p1)
pd.testing.assert_frame_equal(data_v1p1, data_v1p2)
pd.testing.assert_frame_equal(data_v1p2, data_v1p3)

# assert (
#     data_v1p0.equals(data_v1p1) and data_v1p1.equals(data_v1p2) and data_v1p2.equals(data_v1p3)
# )  # Alternative

## Write to file

Let's conclude our discussion of `DataFrame` joining and merging by writing `data_v1p0` to a CSV file. We will make use of the data file in a future lesson. 🙂

In [22]:
filepath = "./data/mega_millions_pick5_cols-2002_24.csv"
data_v1p0.to_csv(filepath, index=False)